# Python Variable Scope

> 📘 **Python Mastery** · Module 03 — Functions · Lesson 4/5

Where does a variable live, and who is allowed to see it? Scope answers both questions. Learn the LEGB lookup rule, why `count += 1` inside a function explodes while `print(count)` works fine, and meet `global`, `nonlocal`, and the closures they unlock.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- **Distinguish** local scope from global scope (and predict `NameError`s)
- **Trace** name lookups using the LEGB rule in nested functions
- **Explain** why reading a global works but assigning to it raises `UnboundLocalError`
- **Use** `global` and `nonlocal` deliberately — and know when to avoid them
- **Recognise** closures: functions that remember their enclosing variables
- **Inspect** namespaces with `globals()` and `locals()`

## 1. Local vs Global Scope

A variable assigned **inside** a function is *local*: it exists only while that function runs, invisible from outside. A variable assigned at the top level of the file is *global*: visible everywhere — including *inside* functions, where it can be read.

Think of locals as sticky notes on a whiteboard, wiped clean the moment the meeting (the function call) ends; globals are carved into the wall.

**Syntax:**

```python
global_var = 10             # global scope: the whole module sees it

def function():
    local_var = 5           # local scope: born and dies inside the function
    print(global_var)       # globals are READABLE from inside
```

**Example:**

In [1]:
shop_name = "Dhaka Delights"       # GLOBAL variable

def print_stock():
    stock = 42                     # LOCAL variable
    print(f"{shop_name} has {stock} units in stock")


print_stock()                      # the body can READ the global
print(shop_name)                   # global still visible outside

try:
    print(stock)                   # local vanished when the call ended
except NameError as err:
    print("NameError:", err)

Dhaka Delights has 42 units in stock
Dhaka Delights
NameError: name 'stock' is not defined


## 2. The LEGB Rule

When Python meets a name, it searches scopes in a strict order and stops at the first hit:

1. **L**ocal — names assigned inside the current function
2. **E**nclosing — names in any outer function (for nested `def`s)
3. **G**lobal — top-level names of the module
4. **B**uilt-in — Python's own names: `len`, `print`, `True`…

"Shadowing" is simply a nearer scope hiding a farther one: define `x` locally and the global `x` is invisible inside that function.

**Syntax:**

```python
x = "global"

def outer():
    x = "enclosing"           # level E
    def inner():
        x = "local"           # level L wins over E, G, B
        print(x)
    inner()
```

**Example:** watch each layer resolve.

In [2]:
x = "global x"

def outer():
    x = "enclosing x"             # level E

    def inner():
        x = "local x"             # level L: nearest hit stops the search
        print("inner sees:", x)

    inner()
    print("outer sees:", x)       # outer has no LOCAL x... but enclosing does

outer()
print("module sees:", x)          # global untouched by either function

# Mental exercise: delete inner's x -> inner would see the ENCLOSING one.
# Delete outer's too -> both would see the GLOBAL one.
# Delete the global as well -> Python finally checks BUILT-INS (and fails).

inner sees: local x
outer sees: enclosing x
module sees: global x


In [3]:
# Level B bites back: shadowing a built-in INSIDE a function
def broken_sum(numbers):
    sum = numbers[0] + numbers[1]     # assignment makes 'sum' LOCAL (level L)
    return sum(numbers[2:])           # ...so this calls an int! TypeError


try:
    print(broken_sum([1, 2, 3, 4]))
except TypeError as err:
    print("TypeError:", err)

print(sum([1, 2, 3]))     # module-level built-in was never touched

TypeError: 'int' object is not callable
6


## 3. Reading vs Assigning Globals

The rules are asymmetric — memorise this pair:

- **Reading** a global from inside a function: fine. The local search finds nothing and walks outward.
- **Assigning** to that name anywhere in the function: Python declares it local **for the whole function body**. So `count += 1` must read the old value of a *local* `count` that does not exist yet → `UnboundLocalError`.

> 🔍 **Under the Hood:** at **compile time** — before any line runs — CPython scans the function body for every name that gets assigned anywhere in it and records them in `co_varnames` as locals. One assignment anywhere makes the name local everywhere in that function, even on lines *above* the assignment.

**Syntax:**

```python
count = 100

def reader():
    print(count)        # READ: allowed, walks out to global

def bumper():
    count += 1          # ASSIGN: count is local -> UnboundLocalError
```

**Example:**

In [4]:
count = 100                       # global

def show_count():
    print(count)                  # READ: allowed


def bump_count():
    count += 1                    # ASSIGN: makes count local for the WHOLE body


show_count()

try:
    bump_count()
except UnboundLocalError as err:
    print("UnboundLocalError:", err)

100
UnboundLocalError: cannot access local variable 'count' where it is not associated with a value


In [5]:
def tricky():
    # print(count)     # <- even THIS line would fail: count is local from line 1
    count = 5          # this assignment claims the name for the entire body
    print(count)


tricky()
print(tricky.__code__.co_varnames)    # compiled-in list of local names

5
('count',)


## 4. The `global` Keyword (and Why to Be Careful)

Declaring `global balance` inside a function tells Python: assignments to this name go to the **module-level** variable. It works — and it is usually the wrong tool.

Functions that mutate globals create invisible links between distant parts of a program: change one function and a seemingly unrelated one misbehaves. Prefer receiving input through parameters and sending results through `return` (Lesson 3!).

**Syntax:**

```python
counter = 0

def tick():
    global counter     # opt in to mutating the MODULE variable
    counter += 1
```

**Example:**

In [6]:
balance = 5000                    # global account balance

def deposit_global(amount):
    global balance                # allowed, but creates a hidden dependency
    balance += amount
    return balance


deposit_global(1500)
print(balance)                    # changed OUTSIDE the function too -- side effect!

# Preferred style: pass state in, get the new state back
def deposit_clean(balance, amount):
    return balance + amount


balance = deposit_clean(balance, 2000)
print(balance)
print(deposit_clean.__code__.co_varnames)   # no surprises hidden inside

6500
8500
('balance', 'amount')


## 5. `nonlocal`: Reaching the Enclosing Scope

Nested functions hit the same wall one level down. `nonlocal name` points at the nearest **enclosing function's** variable — not the global one. It is the sanctioned way for an inner function to update its outer function's locals: the engine behind counters, accumulators and memoisation caches.

**Syntax:**

```python
def outer():
    hits = 0
    def inner():
        nonlocal hits     # target OUTER's hits, not a global
        hits += 1
    inner()
    return hits
```

**Example:**

In [7]:
def make_counter():
    hits = 0                          # lives in make_counter's scope
    def tick():
        nonlocal hits                 # update the ENCLOSING variable
        hits += 1
        return hits
    return tick


page_views = make_counter()
print(page_views(), page_views(), page_views())   # remembers between calls

1 2 3


In [8]:
# Without nonlocal, the same code explodes -- exactly like globals did
def make_counter_broken():
    hits = 0
    def tick():
        hits += 1        # no declaration -> hits is LOCAL to tick -> boom
        return hits
    return tick


broken_counter = make_counter_broken()
try:
    broken_counter()
except UnboundLocalError as err:
    print("UnboundLocalError:", err)

# Vocabulary check:
# global x   -> climbs all the way to MODULE level
# nonlocal x -> stops at the nearest ENCLOSING FUNCTION level

UnboundLocalError: cannot access local variable 'hits' where it is not associated with a value


## 6. Closures: Functions That Remember

Look again at `make_counter`: `make_counter` had **finished**, yet `tick` still used `hits`. An inner function that carries references to its enclosing variables is a **closure** — the enclosing scope stays alive for as long as the inner function needs it.

> 🔍 **Under the Hood:** the captured variables ride along inside the function object in `__closure__` — a tuple of *cell* objects, each holding one captured value. Two counters built by the same factory carry two independent cells, so their states never mix.

**Syntax:**

```python
def make_greeter(greeting):
    def greet(name):              # uses 'greeting' from the enclosing scope
        return f"{greeting}, {name}!"
    return greet

hello = make_greeter("Hello")     # closure: keeps 'greeting' alive
```

**Example:**

In [9]:
def make_greeter(greeting):
    """Return a function that remembers 'greeting' forever."""
    def greet(name):
        return f"{greeting}, {name}!"
    return greet


hello = make_greeter("Hello")
assalamu = make_greeter("Assalamu alaikum")

print(hello("Sarah"))
print(assalamu("Rahim"))

# Peeking inside the closure:
print(len(hello.__closure__), hello.__closure__[0].cell_contents)

Hello, Sarah!
Assalamu alaikum, Rahim!
1 Hello


In [10]:
def make_account(start):
    balance = start
    def spend(amount):
        nonlocal balance
        balance -= amount
        return balance
    return spend


sarah_acct = make_account(1000)
rahim_acct = make_account(50)      # completely separate closure cells

print(sarah_acct(300))     # 700
print(rahim_acct(20))      # 30
print(sarah_acct(100))     # 600 -- Rahim's spending never touched Sarah's cell

700
30
600


## 7. `globals()` and `locals()`

Both return dictionaries mapping names to values. `globals()` shows the module's live namespace (the same everywhere); `locals()` shows whatever scope you call it from — inside a function, that is its locals. Great for debugging "where did this name come from?"

Editing `globals()["name"]` genuinely changes the module. Powerful — and dangerous enough to treat as a last-resort tool.

**Syntax:**

```python
globals()   # dict of module-level names
locals()    # dict of current-scope names (inside a function: its locals)
```

**Example:**

In [11]:
course = "Python Mastery"
lesson_no = 4

def peek():
    topic = "scope"
    print("locals() inside the function:", sorted(locals()))
    print("'topic' visible locally:", "topic" in locals())


peek()

print("'course' in globals():", "course" in globals())
print("Number of module-level names:", len(globals()))

globals()["bonus_note"] = "injected!"     # possible... but please avoid
print(bonus_note)

locals() inside the function: ['topic']
'topic' visible locally: True
'course' in globals(): True
Number of module-level names: 58
injected!


## 8. Scope Lifetime

Locals are **born on assignment and destroyed when the function ends** — every call starts from scratch. That is why an innocent-looking counter fails: each call re-creates `votes = 0`.

Globals live for the entire program run. Closure cells are the beautiful exception: they live as long as the function that captured them — which is exactly why factories in the previous section work.

One more surprise for people coming from C or Java: Python has **no block scope**. Variables created inside `if` or `for` blocks belong to the surrounding function (or module) and survive the block.

**Syntax:**

```python
def f():
    temp = 1          # destroyed when f returns

for i in range(3):
    pass
print(i)              # still alive! i leaked out of the loop
```

**Example:**

In [12]:
def vote_once():
    votes = 0            # re-created EVERY call
    votes += 1
    return votes


print(vote_once(), vote_once(), vote_once())     # always 1 -- no memory


# Compare: a persistent closure-based counter
def make_vote_counter():
    total = 0
    def vote():
        nonlocal total
        total += 1
        return total
    return vote


real_poll = make_vote_counter()
print(real_poll(), real_poll(), real_poll())     # 1 2 3 -- state persists

1 1 1
1 2 3


In [13]:
# No block scope in Python:
for i in range(3):
    doubled = i * 2

print(i)         # 2 -- survived the loop
print(doubled)   # 4 -- so did this

# Inside functions the same leak happens, but locals still die with the call:
def loopy():
    for j in range(2):
        message = f"pass {j}"
    return message        # legal: j and message belong to loopy's scope

print(loopy())

try:
    print(j)              # ...but loopy's locals never reach out here
except NameError as err:
    print("NameError:", err)

2
4
pass 1
NameError: name 'j' is not defined


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
| --- | --- | --- |
| Assigning to a global's name expecting an update | `UnboundLocalError` — assignment claims the name locally | Pass the value in and `return` the new one (or `nonlocal` in nested helpers) |
| Shadowing built-ins (`sum`, `list`, `id`) | Later calls crash with baffling errors | Never reuse built-in names; pick descriptive plurals |
| Expecting block scope around `if`/`for` | Loop variables leak into the function/module and linger | Rename deliberately; don't rely on block boundaries |
| Many functions mutating one shared global | Order-dependent bugs; impossible to test in isolation | Keep state in parameters, return values, or closures/classes |
| Editing the `globals()` dict casually | Spooky action at a distance; typos create rogue variables | Explicit parameters beat namespace surgery every time |

## 💡 Best Practices & Pro Tips

- Default to **pure functions**: everything enters via parameters, everything leaves via `return`. Globals only for genuine constants (`MAX_RETRIES = 3`, written in UPPER_CASE).
- If two functions share mutable state, wrap the state and the functions together (a class) rather than threading a global through five files.
- Reserve `global`/`nonlocal` for the rare legitimate case — counters and caches inside factories — and comment why.
- When debugging, `print(sorted(locals()))` inside a function instantly shows what it actually has to work with.
- 🤖 **AI-engineering relevance:** data-science libraries avoid hidden globals because pipelines must be reproducible — the same input must give the same output. The closest cousin you will meet is library-level *configuration state* (e.g., NumPy random seeds, pandas options): set it deliberately, once, at the top of a notebook, and note that it behaves like a global for everything below.

## 📌 Summary

| Tool | What it does | Example |
| --- | --- | --- |
| local assignment | Name exists only during the call | `stock = 42` inside a function |
| global read | Functions may read module names freely | `print(shop_name)` |
| `global x` | Assignments target the module variable | `global balance; balance += a` |
| `nonlocal x` | Assignments target the nearest enclosing function | `nonlocal hits; hits += 1` |
| LEGB | Lookup order: Local → Enclosing → Global → Built-in | shadowing resolved nearest-first |
| `__closure__` | Cells holding captured enclosing variables | `f.__closure__[0].cell_contents` |
| `globals()` / `locals()` | Namespace dicts for inspection | `sorted(locals())` |
| co-varnames | Compiled list of local names | `f.__code__.co_varnames` |

Key takeaways:

- Assignment creates a local — anywhere in the body means everywhere in the body.
- LEGB is the whole lookup story; first hit wins.
- `global`/`nonlocal` work but couple code invisibly — parameters and return values scale better.
- Locals die when the call ends; closures are the sanctioned way to let a function remember.

## 🔗 Next Lesson

➡️ Finish Module 03 with **05_Lambda** — anonymous one-line functions and their natural habitats: `sorted(key=...)`, `map()` and `filter()`.